## Setup and derived features
squad_value_diff and form_diff aren't new information — they're linear combinations of columns already in the table — but giving a tree model an explicit "advantage" feature directly usually lets it find a good split in fewer levels than forcing it to compare two separate columns implicitly.

In [0]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer, VectorAssembler, VectorIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

CATALOG = "workspace"
SCHEMA = "transfermarkt"

data = spark.read.parquet(f"/Volumes/{CATALOG}/{SCHEMA}/ml/classification_features_final")

data = (data
    .withColumn("squad_value_diff", F.col("home_squad_value_eur") - F.col("away_squad_value_eur"))
    .withColumn("form_diff", F.col("home_rolling_win_rate") - F.col("away_rolling_win_rate")))

data.groupBy("year").count().orderBy("year").show()

## Time-based split, not a random one
This is forecasting a future result from past information, not i.i.d. classification — a random 80/20 split would let the model train on 2023 matches and get tested on 2013 ones, which is never how it would actually be used and can flatter performance if there's any drift over time. Train on the oldest years, tune on the middle years, touch the most recent year exactly once for final evaluation.

In [0]:
train = data.filter(F.col("year") <= 2020)
val = data.filter((F.col("year") >= 2021) & (F.col("year") <= 2022))
test = data.filter(F.col("year") == 2023)

print("train:", train.count(), " val:", val.count(), " test:", test.count())

## The real baseline
The bar to beat is this scoped dataset's own majority-class rate on the actual test set — not the 45.1% population figure from the original EDA, for the reason discussed above.

In [0]:
majority_class = train.groupBy("result").count().orderBy(F.desc("count")).first()["result"]
test_majority_acc = test.filter(F.col("result") == majority_class).count() / test.count()

print("Majority class:", majority_class)
print(f"Majority-class baseline accuracy on test: {test_majority_acc:.3f}")
test.groupBy("result").count().withColumn("pct", F.round(F.col("count") / test.count() * 100, 1)).show()

## Feature preparation
competition_key is a surrogate key, not a continuous quantity — feeding it in raw would wrongly imply competition #40 is "greater than" competition #12. VectorIndexer inspects each assembled column and auto-flags any with fewer than maxCategories distinct values as categorical (competition_key has ~65 — well under 100), so it gets one-hot-style tree splits instead of being treated as ordinal. Both this and the label indexer are fit on the full dataset rather than train-only — that's safe, not leakage, because neither one ever looks at the label; they only need to see every category value that exists so val/test don't crash on a category train never saw.

In [0]:
label_indexer = StringIndexer(inputCol="result", outputCol="label")
label_model = label_indexer.fit(data)
print("Label mapping:", list(enumerate(label_model.labels)))

continuous_cols = ["home_rolling_win_rate", "away_rolling_win_rate",
                    "home_squad_value_eur", "away_squad_value_eur",
                    "squad_value_diff", "form_diff"]

continuous_assembler = VectorAssembler(inputCols=continuous_cols, outputCol="continuous_features")

# competition_key gets its own isolated assembler+indexer, so it's the ONLY thing that
# can ever be tagged categorical — a continuous column with low cardinality (like form_diff)
# can no longer get swept up by VectorIndexer's arity guess
competition_assembler = VectorAssembler(inputCols=["competition_key"], outputCol="competition_raw")
competition_indexer = VectorIndexer(inputCol="competition_raw", outputCol="competition_indexed", maxCategories=100)

final_assembler = VectorAssembler(inputCols=["continuous_features", "competition_indexed"], outputCol="features")

feature_cols = continuous_cols + ["competition_key"]  # order matches the concatenated vector below

def prep(df):
    df = label_model.transform(df)
    df = continuous_assembler.transform(df)
    df = competition_assembler.transform(df)
    df = competition_indexer.transform(df)
    df = final_assembler.transform(df)
    return df

comp_fit_model = competition_indexer.fit(competition_assembler.transform(data))
competition_indexer = comp_fit_model  # use the fitted version from here on

train_prepped = prep(train)
val_prepped = prep(val)

## Balancing the training set, and a metric that can't hide draw failure
balance_classes oversamples minority classes up to the majority count, with replacement — applied only to training data, never to val or test, which must stay the real distribution for evaluation to mean anything. macro_f1 averages per-class F1 equally regardless of class size, so a model that ignores draws can no longer look fine just because it's good at the majority classes.

In [0]:
def balance_classes(df, label_col="result"):
    counts = {row[label_col]: row["count"] for row in df.groupBy(label_col).count().collect()}
    max_count = max(counts.values())
    parts = []
    for cls, count in counts.items():
        part = df.filter(F.col(label_col) == cls)
        fraction = max_count / count
        if fraction > 1.0:
            part = part.sample(withReplacement=True, fraction=fraction, seed=42)
        parts.append(part)
    balanced = parts[0]
    for p in parts[1:]:
        balanced = balanced.union(p)
    return balanced

def macro_f1(predictions):
    scores = []
    for label in [0.0, 1.0, 2.0]:
        ev = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                                 metricName="fMeasureByLabel", metricLabel=label)
        scores.append(ev.evaluate(predictions))
    return sum(scores) / len(scores), scores

print("Before balancing:")
train.groupBy("result").count().show()

train_balanced = balance_classes(train)
print("After balancing:")
train_balanced.groupBy("result").count().show()

train_prepped = prep(train_balanced)
# val_prepped is untouched on purpose — real distribution, needed for honest evaluation

## Hyperparameter search on the validation set
A small manual grid, not Spark's built-in CrossValidator — CrossValidator does random k-fold splits by default, which would reintroduce the exact temporal leakage the time-based split was built to avoid. Selecting by weighted F1 rather than accuracy, since the classes aren't balanced.

In [0]:
import mlflow

mlflow.set_experiment("/Shared/transfermarkt_classification")

best_macro_f1, best_params = -1, None
for num_trees in [50, 100]:
    for max_depth in [5, 10]:
        with mlflow.start_run(run_name=f"rf_trees{num_trees}_depth{max_depth}"):
            rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                                         numTrees=num_trees, maxDepth=max_depth, maxBins=100, seed=42)
            model = rf.fit(train_prepped)
            mf1, (home_f1, away_f1, draw_f1) = macro_f1(model.transform(val_prepped))

            mlflow.log_params({"numTrees": num_trees, "maxDepth": max_depth, "maxBins": 100})
            mlflow.log_metrics({"val_macro_f1": mf1, "val_home_f1": home_f1,
                                 "val_away_f1": away_f1, "val_draw_f1": draw_f1})

            print(f"numTrees={num_trees}, maxDepth={max_depth}: macro F1={mf1:.4f} "
                  f"(home={home_f1:.3f}, away={away_f1:.3f}, draw={draw_f1:.3f})")
            if mf1 > best_macro_f1:
                best_macro_f1, best_params = mf1, (num_trees, max_depth)

print("Best params:", best_params, " val macro F1:", best_macro_f1)

## Final model — train on train+val, touch test exactly once
Retrains with the winning hyperparameters on the combined train+validation data, then evaluates on test a single time. Reporting the confusion matrix alongside accuracy, since a good overall number can still hide a model that never correctly predicts draws — a well-known hard class in football outcome prediction.

In [0]:
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
evaluator_f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")

train_val_balanced = balance_classes(train.union(val))
train_val_prepped = prep(train_val_balanced)
test_prepped = prep(test)

with mlflow.start_run(run_name="final_model"):
    final_rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                                       numTrees=best_params[0], maxDepth=best_params[1], maxBins=100, seed=42)
    final_model = final_rf.fit(train_val_prepped)
    test_preds = final_model.transform(test_prepped)

    test_acc = evaluator_acc.evaluate(test_preds)
    test_weighted_f1 = evaluator_f1.evaluate(test_preds)
    test_mf1, (home_f1, away_f1, draw_f1) = macro_f1(test_preds)

    mlflow.log_params({"numTrees": best_params[0], "maxDepth": best_params[1], "maxBins": 100})
    mlflow.log_metrics({
        "test_accuracy": test_acc, "test_weighted_f1": test_weighted_f1,
        "test_macro_f1": test_mf1, "test_home_f1": home_f1,
        "test_away_f1": away_f1, "test_draw_f1": draw_f1,
        "majority_baseline_accuracy": test_majority_acc,
    })
    mlflow.spark.log_model(final_model, "model",
                            dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/ml/mlflow_tmp")

    print("Test accuracy:", test_acc)
    print("Test weighted F1:", test_weighted_f1)
    print(f"Test macro F1: {test_mf1:.4f}  (home={home_f1:.3f}, away={away_f1:.3f}, draw={draw_f1:.3f})")
    print("Majority-class baseline accuracy:", test_majority_acc)

test_preds.groupBy("label", "prediction").count().orderBy("label", "prediction").show(20)

## Feature importances

In [0]:
for name, score in sorted(zip(feature_cols, final_model.featureImportances.toArray()), key=lambda x: -x[1]):
    print(f"{name}: {score:.4f}")

## Walk-forward validation across 3 folds
Fixed hyperparameters (from the grid search), retrained on an expanding window, tested on the single immediately-following year each time. 2021/2022 folds aren't fully clean — those years were part of the original validation set — so treat them as a stability check on the fixed recipe, not an independent re-validation. 2023 replicates the already-clean test result.

In [0]:
def evaluate_fold(train_end_year, test_year):
    fold_train = data.filter(F.col("year") <= train_end_year)
    fold_test = data.filter(F.col("year") == test_year)

    fold_train_balanced = balance_classes(fold_train)
    fold_train_prepped = prep(fold_train_balanced)
    fold_test_prepped = prep(fold_test)

    fold_majority = fold_train.groupBy("result").count().orderBy(F.desc("count")).first()["result"]
    fold_baseline_acc = fold_test.filter(F.col("result") == fold_majority).count() / fold_test.count()

    rf = RandomForestClassifier(labelCol="label", featuresCol="features",
                                 numTrees=best_params[0], maxDepth=best_params[1], maxBins=100, seed=42)
    model = rf.fit(fold_train_prepped)
    preds = model.transform(fold_test_prepped)

    acc = evaluator_acc.evaluate(preds)
    mf1, (home_f1, away_f1, draw_f1) = macro_f1(preds)

    return {"test_year": test_year, "train_rows": fold_train.count(), "test_rows": fold_test.count(),
            "baseline_acc": fold_baseline_acc, "accuracy": acc, "macro_f1": mf1,
            "home_f1": home_f1, "away_f1": away_f1, "draw_f1": draw_f1}

for train_end, test_yr in [(2020, 2021), (2021, 2022), (2022, 2023)]:
    r = evaluate_fold(train_end, test_yr)
    print(f"Test year {r['test_year']}: train_rows={r['train_rows']}, test_rows={r['test_rows']}, "
          f"baseline_acc={r['baseline_acc']:.3f}, accuracy={r['accuracy']:.3f}, macro_f1={r['macro_f1']:.3f} "
          f"(home={r['home_f1']:.3f}, away={r['away_f1']:.3f}, draw={r['draw_f1']:.3f})")

## A single artifact for real inference on new data
PipelineModel wraps the already-fitted prep steps and classifier together — no refitting, reuses exactly what Cells 4-6 already produced. label_model is deliberately left out here (the true outcome isn't known at inference time); IndexToString takes its place at the end to turn the numeric prediction back into a readable result. SQLTransformer computes squad_value_diff/form_diff internally, so raw input only needs the 5 genuinely raw columns.

In [0]:
from pyspark.ml import PipelineModel
from pyspark.ml.feature import IndexToString, SQLTransformer

diff_transformer = SQLTransformer(statement="""
    SELECT *,
           (home_squad_value_eur - away_squad_value_eur) AS squad_value_diff,
           (home_rolling_win_rate - away_rolling_win_rate) AS form_diff
    FROM __THIS__
""")

inference_pipeline = PipelineModel(stages=[
    diff_transformer,
    continuous_assembler,
    competition_assembler,
    competition_indexer,
    final_assembler,
    final_model,
    IndexToString(inputCol="prediction", outputCol="predicted_result", labels=label_model.labels),
])

with mlflow.start_run(run_name="inference_pipeline"):
    mlflow.spark.log_model(inference_pipeline, "inference_pipeline",
                            dfs_tmpdir=f"/Volumes/{CATALOG}/{SCHEMA}/ml/mlflow_tmp")

# Smoke test — made-up example: strong home form/value advantage
made_up = spark.createDataFrame([
    (0.8, 0.4, 400_000_000.0, 150_000_000.0, 30),
], ["home_rolling_win_rate", "away_rolling_win_rate", "home_squad_value_eur",
    "away_squad_value_eur", "competition_key"])

inference_pipeline.transform(made_up).select("predicted_result", "probability").show(truncate=False)